In [ ]:
import pandas as pd
import sys
import os
from xgboost import XGBClassifier

sys.path.append(os.path.abspath(".."))

from src.data import get_xy
from src.models import evaluate_model, print_results

In [ ]:
df_train = pd.read_excel("training_barcelona_shots.xlsx", header=0)
df_test = pd.read_excel("testing_barcelona_shots.xlsx", header=0)

shot_features = [
    "distance_d",
    "angle_d",
    "free_kick_flag",
    "penalty_flag",
    "technique_b",
    "n_def_1_5",
    "n_def_3_0",
    "dist_nearest_def",
    "gk_dist_to_shooter"
]

X_train, y_train = get_xy(df_train, shot_features)
X_test, y_test = get_xy(df_test, shot_features)

In [ ]:
model = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 1, 5]
}

In [ ]:
grid, results = evaluate_model(model, param_grid, X_train, y_train)
print_results(results)

In [ ]:
final_model = XGBClassifier(
    **grid.best_params_,
    eval_metric="logloss",
    random_state=42
)

final_model.fit(X_train, y_train)

y_pred_proba_xgb = final_model.predict_proba(X_test)[:, 1]